<a href="https://colab.research.google.com/github/yiqi0806-collab/fitbit-sleep-quality/blob/main/RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import shutil

os.makedirs("/content/data", exist_ok=True)

pdf_path = "/content/Yue_Okten_revision.pdf"
target_path = "/content/data/Yue_Okten_revision.pdf"

if os.path.exists(pdf_path):
    shutil.move(pdf_path, target_path)

print("Files in data folder:")
print(os.listdir("/content/data"))

Files in data folder:
[]


In [ ]:
!pip install pypdf
!pip install sentence-transformers
!pip install scikit-learn
!pip install openai
import os
import numpy as np
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity



# 如果你想用 OpenAI 生成答案，取消下面两行注释
# from openai import OpenAI
# client = OpenAI(api_key="你的API_KEY")


# =========================
# 1. 读取 PDF 或 TXT 文件
# =========================

def read_pdf(file_path):
    reader = PdfReader(file_path)
    text = ""

    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text + "\n"

    return text


def read_txt(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        return f.read()


def load_documents(data_folder):
    documents = []

    for filename in os.listdir(data_folder):
        file_path = os.path.join(data_folder, filename)

        if filename.endswith(".pdf"):
            text = read_pdf(file_path)
        elif filename.endswith(".txt"):
            text = read_txt(file_path)
        else:
            continue

        documents.append({
            "filename": filename,
            "text": text
        })

    return documents


# =========================
# 2. 切 chunk
# =========================

def chunk_text(text, chunk_size=500, overlap=100):
    """
    chunk_size: 每个 chunk 大约多少个单词
    overlap: 相邻 chunk 之间重叠多少个单词
    """

    words = text.split()
    chunks = []

    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


def build_chunks(documents):
    all_chunks = []

    for doc in documents:
        chunks = chunk_text(doc["text"])

        for i, chunk in enumerate(chunks):
            all_chunks.append({
                "filename": doc["filename"],
                "chunk_id": i,
                "text": chunk
            })

    return all_chunks


# =========================
# 3. 建 embedding index
# =========================

def build_embedding_index(chunks, embedding_model):
    texts = [chunk["text"] for chunk in chunks]

    embeddings = embedding_model.encode(texts)

    return np.array(embeddings)


# =========================
# 4. 根据问题检索相关 chunk
# =========================

def retrieve(query, chunks, chunk_embeddings, embedding_model, top_k=3):
    query_embedding = embedding_model.encode([query])

    similarities = cosine_similarity(query_embedding, chunk_embeddings)[0]

    top_indices = np.argsort(similarities)[::-1][:top_k]

    results = []

    for idx in top_indices:
        results.append({
            "filename": chunks[idx]["filename"],
            "chunk_id": chunks[idx]["chunk_id"],
            "text": chunks[idx]["text"],
            "score": similarities[idx]
        })

    return results


# =========================
# 5. 把检索结果交给 LLM
# =========================

def build_prompt(query, retrieved_chunks):
    context = ""

    for i, chunk in enumerate(retrieved_chunks):
        context += f"\n[Chunk {i+1} from {chunk['filename']}]\n"
        context += chunk["text"]
        context += "\n"

    prompt = f"""
You are a helpful research assistant.

Use the following context to answer the question.
If the answer is not in the context, say that the papers do not provide enough information.

Context:
{context}

Question:
{query}

Answer:
"""

    return prompt


def ask_llm(prompt):
    """
    这里是生成答案的部分。
    你可以先不管它，只看 RAG 的 retrieval 部分。
    """

    # 如果你想用 OpenAI API，可以取消下面这段注释
    """
    response = client.chat.completions.create(
        model="你的模型名称",
        messages=[
            {"role": "user", "content": prompt}
        ]
    )

    return response.choices[0].message.content
    """

    # 先返回 prompt，方便你看 RAG 到底塞了什么东西给模型
    return prompt


# =========================
# 6. 主程序
# =========================

def main():
    data_folder = "data"

    print("Loading documents...")
    documents = load_documents(data_folder)

    print(f"Loaded {len(documents)} documents.")

    print("Building chunks...")
    chunks = build_chunks(documents)

    print(f"Created {len(chunks)} chunks.")

    print("Loading embedding model...")
    embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

    print("Building embedding index...")
    chunk_embeddings = build_embedding_index(chunks, embedding_model)

    while True:
        query = input("\nAsk a question, or type 'exit': ")

        if query.lower() == "exit":
            break

        retrieved_chunks = retrieve(
            query=query,
            chunks=chunks,
            chunk_embeddings=chunk_embeddings,
            embedding_model=embedding_model,
            top_k=3
        )

        print("\nMost relevant chunks:")
        for r in retrieved_chunks:
            print("----------------------")
            print("File:", r["filename"])
            print("Chunk ID:", r["chunk_id"])
            print("Score:", r["score"])
            print(r["text"][:500])

        prompt = build_prompt(query, retrieved_chunks)

        answer = ask_llm(prompt)

        print("\nFinal prompt / answer:")
        print(answer)


if __name__ == "__main__":
    main()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 9.1 MB/s eta 0:00:00
Loading documents...
Loaded 0 documents.
Building chunks...
Created 0 chunks.
Loading embedding model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Building embedding index...

Ask a question, or type 'exit': What is the proposed method in this paper?


ValueError: Expected 2D array, got 1D array instead:
array=[].
Reshape your data either using array.reshape(-1, 1) if your data has a single feature or array.reshape(1, -1) if it contains a single sample.